**Input embeddings create**

In [57]:
import torch

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your (x^1)
        [0.55, 0.87, 0.66],  # journey (x^2)
        [0.57, 0.85, 0.64],  # starts (x^3)
        [0.22, 0.58, 0.33],  # with (x^4)
        [0.77, 0.25, 0.10],  # one (x^5)
        [0.05, 0.80, 0.55]   # step (x^6)
    ]
)

print(inputs)
print("Shape:", inputs.shape)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
Shape: torch.Size([6, 3])


**Query select**

In [58]:
query = inputs[1]

print(query)
print("Shape:", query.shape)

tensor([0.5500, 0.8700, 0.6600])
Shape: torch.Size([3])


**dot product Query and input embedding**

In [59]:
attn_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


**Dot product manually**

In [60]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print("Manual result:", res)
print("PyTorch result:", torch.dot(inputs[0], query))

Manual result: tensor(0.9544)
PyTorch result: tensor(0.9544)


**Simple normalization**

In [61]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print(attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


**Softmax manually implement**

In [62]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


**PyTorch Softmax**

In [63]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


**Single context vector**

In [64]:
query = inputs[1]

context_vec_2 = torch.zeros(query.shape)

for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print("Context vector:", context_vec_2)
print("Shape:", context_vec_2.shape)

Context vector: tensor([0.4419, 0.6515, 0.5683])
Shape: torch.Size([3])


**All tokens attention scores calculate**

In [65]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)
print("Shape:", attn_scores.shape)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
Shape: torch.Size([6, 6])


**Matrix multiplication attention scores calculate**

In [66]:
attn_scores = inputs @ inputs.T

print(attn_scores)
print("Shape:", attn_scores.shape)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
Shape: torch.Size([6, 6])


**Attention weights calculate**

In [67]:
attn_weights = torch.softmax(attn_scores, dim=1)

print(attn_weights)
print("Shape:", attn_weights.shape)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
Shape: torch.Size([6, 6])


**Check row sum**

In [68]:
print("All row sums:", attn_weights.sum(dim=1))

All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


**All context vectors calculate**

In [69]:
all_context_vecs = attn_weights @ inputs

print(all_context_vecs)
print("Shape:", all_context_vecs.shape)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
Shape: torch.Size([6, 3])


**Compare**

In [70]:
print("Individual context vector:")
print(context_vec_2)

print("\nContext vector from matrix:")
print(all_context_vecs[1])

print("\nAre they equal?")
print(torch.allclose(context_vec_2, all_context_vecs[1]))

Individual context vector:
tensor([0.4419, 0.6515, 0.5683])

Context vector from matrix:
tensor([0.4419, 0.6515, 0.5683])

Are they equal?
True


**3.4**

**Input embeddings**

In [71]:
import torch
import torch.nn as nn

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your
        [0.55, 0.87, 0.66],  # journey
        [0.57, 0.85, 0.64],  # starts
        [0.22, 0.58, 0.33],  # with
        [0.77, 0.25, 0.10],  # one
        [0.05, 0.80, 0.55]   # step
    ]
)

print(inputs)
print("Shape:", inputs.shape)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
Shape: torch.Size([6, 3])


**Define dimensions**

In [72]:
x_2 = inputs[1]

d_in = inputs.shape[1]
d_out = 2

print("Second input:", x_2)
print("Input dimension:", d_in)
print("Output dimension:", d_out)

Second input: tensor([0.5500, 0.8700, 0.6600])
Input dimension: 3
Output dimension: 2


**Initialize trainable weight matrices**

In [73]:
torch.manual_seed(123)

W_query = nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

W_key = nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

W_value = nn.Parameter(
    torch.rand(d_in, d_out),
    requires_grad=False
)

print("W_query:\n", W_query)
print("W_key:\n", W_key)
print("W_value:\n", W_value)

W_query:
 Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
W_key:
 Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])
W_value:
 Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


**Calculate Query, Key and Value**

In [74]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print("Query:", query_2)
print("Key:", key_2)
print("Value:", value_2)

Query: tensor([0.4306, 1.4551])
Key: tensor([0.4433, 1.1419])
Value: tensor([0.3951, 1.0037])


**Calculate Keys and Values for all tokens**

In [75]:
keys = inputs @ W_key
values = inputs @ W_value

print("Keys:")
print(keys)
print("Keys shape:", keys.shape)

print("\nValues:")
print(values)
print("Values shape:", values.shape)

Keys:
tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
Keys shape: torch.Size([6, 2])

Values:
tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])
Values shape: torch.Size([6, 2])


**Calculate one attention score**

In [76]:
keys_2 = keys[1]

attn_score_22 = query_2.dot(keys_2)

print("Attention score:", attn_score_22)

Attention score: tensor(1.8524)


**Calculate all attention scores for one query**

In [77]:
attn_scores_2 = query_2 @ keys.T

print("Attention scores:", attn_scores_2)
print("Shape:", attn_scores_2.shape)

Attention scores: tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])
Shape: torch.Size([6])


**Calculate scaled attention weights**

In [78]:
d_k = keys.shape[-1]

attn_weights_2 = torch.softmax(
    attn_scores_2 / d_k**0.5,
    dim=-1
)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
Sum: tensor(1.)


**Calculate context vector**

In [79]:
context_vec_2 = attn_weights_2 @ values

print("Context vector:", context_vec_2)
print("Shape:", context_vec_2.shape)

Context vector: tensor([0.3061, 0.8210])
Shape: torch.Size([2])


**SelfAttention_v1 class**

In [80]:
class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()

        self.d_out = d_out

        self.W_query = nn.Parameter(
            torch.rand(d_in, d_out)
        )

        self.W_key = nn.Parameter(
            torch.rand(d_in, d_out)
        )

        self.W_value = nn.Parameter(
            torch.rand(d_in, d_out)
        )

    def forward(self, x):

        keys = x @ self.W_key

        queries = x @ self.W_query

        values = x @ self.W_value

        attn_scores = queries @ keys.T

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        context_vec = attn_weights @ values

        return context_vec

**Create and run SelfAttention_v1**

In [81]:
torch.manual_seed(123)

sa_v1 = SelfAttention_v1(d_in, d_out)

output_v1 = sa_v1(inputs)

print(output_v1)
print("Output shape:", output_v1.shape)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)
Output shape: torch.Size([6, 2])


**SelfAttention_v2 class using nn.Linear**

In [82]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        self.d_out = d_out

        self.W_query = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )

        self.W_key = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )

        self.W_value = nn.Linear(
            d_in, d_out, bias=qkv_bias
        )

    def forward(self, x):

        keys = self.W_key(x)

        queries = self.W_query(x)

        values = self.W_value(x)

        attn_scores = queries @ keys.T

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        context_vec = attn_weights @ values

        return context_vec

**Create and run SelfAttention_v2**

In [83]:
torch.manual_seed(789)

sa_v2 = SelfAttention_v2(d_in, d_out)

output_v2 = sa_v2(inputs)

print(output_v2)
print("Output shape:", output_v2.shape)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
Output shape: torch.Size([6, 2])


**Transfer weights from v2 to v1**

In [84]:
torch.manual_seed(123)

sa_v1 = SelfAttention_v1(d_in, d_out)

torch.manual_seed(789)

sa_v2 = SelfAttention_v2(d_in, d_out)

with torch.no_grad():

    sa_v1.W_query.copy_(
        sa_v2.W_query.weight.T
    )

    sa_v1.W_key.copy_(
        sa_v2.W_key.weight.T
    )

    sa_v1.W_value.copy_(
        sa_v2.W_value.weight.T
    )

**Compare outputs**

In [85]:
out_v1 = sa_v1(inputs)

out_v2 = sa_v2(inputs)

print("V1 Output:")
print(out_v1)

print("\nV2 Output:")
print(out_v2)

print("\nAre outputs equal?")
print(torch.allclose(out_v1, out_v2))

V1 Output:
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

V2 Output:
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

Are outputs equal?
True


**Section 3.5**

**Query, Key aur Attention Scores**

In [86]:
queries = sa_v2.W_query(inputs)

keys = sa_v2.W_key(inputs)

attn_scores = queries @ keys.T

print("Attention scores:")
print(attn_scores)

Attention scores:
tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)


**Normal Attention Weights**

In [87]:
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5,
    dim=1
)

print("Attention weights:")
print(attn_weights)

Attention weights:
tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


**Simple Lower Triangular Mask**

In [88]:
context_length = attn_scores.shape[0]

mask_simple = torch.tril(
    torch.ones(context_length, context_length)
)

print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


**Mask ko Attention Weights par Apply Karna**

In [89]:
masked_simple = attn_weights * mask_simple

print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


**Masked Weights ko Normalize Karna**

In [90]:
row_sums = masked_simple.sum(
    dim=1,
    keepdim=True
)

masked_simple_norm = masked_simple / row_sums

print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


**Make Efficient Mask**

In [91]:
mask = torch.triu(
    torch.ones(context_length, context_length),
    diagonal=1
)

print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


**Make Future Attention Scores -infinity**

In [92]:
masked = attn_scores.masked_fill(
    mask.bool(),
    -torch.inf
)

print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


**Final Causal Attention Weights**

In [93]:
attn_weights = torch.softmax(
    masked / keys.shape[-1]**0.5,
    dim=1
)

print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


**Context Vectors Calculate**

In [94]:
values = sa_v2.W_value(inputs)

context_vec = attn_weights @ values

print("Context vectors:")
print(context_vec)

print("Context vector shape:", context_vec.shape)

Context vectors:
tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)
Context vector shape: torch.Size([6, 2])


**Dropout Simple Example**

In [95]:
torch.manual_seed(123)

dropout = nn.Dropout(0.5)

example = torch.ones(6, 6)

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


**Attention Weights with Dropout**

In [96]:
torch.manual_seed(123)

print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


**Different Dropout Rate Example***

In [97]:
dropout_small = nn.Dropout(0.1)

print(dropout_small(attn_weights))

tensor([[1.1111, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6130, 0.4981, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3441, 0.3448, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.2734, 0.2736, 0.2576, 0.0000, 0.0000],
        [0.2417, 0.0000, 0.2204, 0.2097, 0.2189, 0.0000],
        [0.2150, 0.1848, 0.1851, 0.1713, 0.1851, 0.1699]],
       grad_fn=<MulBackward0>)


**Batch Input**

In [98]:
batch = torch.stack(
    (inputs, inputs),
    dim=0
)

print(batch.shape)

torch.Size([2, 6, 3])


**CausalAttention Class**

In [99]:
class CausalAttention(nn.Module):

    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        qkv_bias=False
    ):

        super().__init__()

        self.d_out = d_out

        self.W_query = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        self.W_key = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        self.W_value = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):

        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)

        queries = self.W_query(x)

        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)

        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf
        )

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values

        return context_vec

**CausalAttention Object**

In [100]:
torch.manual_seed(123)

context_length = batch.shape[1]

ca = CausalAttention(
    d_in=3,
    d_out=2,
    context_length=context_length,
    dropout=0.0
)

In [101]:
context_vecs = ca(batch)

print("Context vectors:")
print(context_vecs)

print("Context vectors shape:", context_vecs.shape)

Context vectors:
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
Context vectors shape: torch.Size([2, 6, 2])


**3.6**

**MultiHeadAttentionWrapper class**

In [102]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False
    ):
        super().__init__()

        self.heads = nn.ModuleList(
            [
                CausalAttention(
                    d_in,
                    d_out,
                    context_length,
                    dropout,
                    qkv_bias
                )
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):

        return torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

**MultiHeadAttentionWrapper run**

In [103]:
torch.manual_seed(123)

context_length = batch.shape[1]

d_in, d_out = 3, 2

mha = MultiHeadAttentionWrapper(
    d_in,
    d_out,
    context_length,
    0.0,
    num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)

print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


**Exercise 3.2**

In [104]:
mha = MultiHeadAttentionWrapper(
    d_in=3,
    d_out=1,
    context_length=context_length,
    dropout=0.0,
    num_heads=2
)

context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


**MultiHeadAttention class**

In [105]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False
    ):
        super().__init__()

        assert d_out % num_heads == 0, \
            "d_out must be divisible by num_heads"

        self.d_out = d_out

        self.num_heads = num_heads

        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        self.W_key = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        self.W_value = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        self.out_proj = nn.Linear(
            d_out,
            d_out
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):

        b, num_tokens, d_in = x.shape

        # Step 1: Calculate keys, queries and values

        keys = self.W_key(x)

        queries = self.W_query(x)

        values = self.W_value(x)

        # Step 2: Split into multiple heads

        keys = keys.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim
        )

        values = values.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim
        )

        queries = queries.view(
            b,
            num_tokens,
            self.num_heads,
            self.head_dim
        )

        # Step 3: Transpose heads and tokens

        keys = keys.transpose(1, 2)

        queries = queries.transpose(1, 2)

        values = values.transpose(1, 2)

        # Step 4: Calculate attention scores

        attn_scores = queries @ keys.transpose(2, 3)

        # Step 5: Apply causal mask

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(
            mask_bool,
            -torch.inf
        )

        # Step 6: Calculate attention weights

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        # Step 7: Apply dropout

        attn_weights = self.dropout(attn_weights)

        # Step 8: Calculate context vectors

        context_vec = (attn_weights @ values).transpose(1, 2)

        # Step 9: Combine all heads

        context_vec = context_vec.contiguous().view(
            b,
            num_tokens,
            self.d_out
        )

        # Step 10: Output projection

        context_vec = self.out_proj(context_vec)

        return context_vec

**Efficient MultiHeadAttention run**

In [106]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape

d_out = 2

mha = MultiHeadAttention(
    d_in,
    d_out,
    context_length,
    dropout=0.0,
    num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)

print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


**Batched Matrix Multiplication

Example tensor**

In [107]:
a = torch.tensor([
    [
        [
            [0.2745, 0.6584, 0.2775, 0.8573],
            [0.8993, 0.0390, 0.9268, 0.7388],
            [0.7179, 0.7058, 0.9156, 0.4340]
        ],
        [
            [0.0772, 0.3565, 0.1479, 0.5331],
            [0.4066, 0.2318, 0.4545, 0.9737],
            [0.4606, 0.5159, 0.4220, 0.5786]
        ]
    ]
])

print("Shape:", a.shape)

Shape: torch.Size([1, 2, 3, 4])


**Batched matrix multiplication**

In [108]:
print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


**First head separate calculation**

In [109]:
first_head = a[0, 0, :, :]

first_res = first_head @ first_head.T

print("First head:\n", first_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])


**Second head separate calculation**

In [110]:
second_head = a[0, 1, :, :]

second_res = second_head @ second_head.T

print("Second head:\n", second_res)

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


**GPT-2 Size Attention Module**

In [111]:
torch.manual_seed(123)

gpt2_mha = MultiHeadAttention(
    d_in=768,
    d_out=768,
    context_length=1024,
    dropout=0.0,
    num_heads=12
)

print("Number of heads:", gpt2_mha.num_heads)

print("Head dimension:", gpt2_mha.head_dim)

print("Output dimension:", gpt2_mha.d_out)

Number of heads: 12
Head dimension: 64
Output dimension: 768
